# Relaxed heuristics for A*

_Artificial Intelligence — more_

**Drop a rule to make an easier problem. Its cost becomes a heuristic that never lies high.**

A* needs a heuristic $h(s)$: a guess of the remaining cost to the goal. It must never overestimate.

     Where do we get such a guess? Relax the problem: throw away a constraint to make an easier version.

---

This notebook is a practice scaffold for the **Relaxed heuristics for A*** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Build the maze and the relaxed heuristic

A* needs a heuristic $h(s)$ that *never overestimates* the true remaining cost. We get one by **relaxing** the problem: pretend the walls aren't there.

With walls removed, the cheapest path is just the **Manhattan distance** — the sum of the horizontal and vertical gaps. Because removing constraints can only make a problem easier, this relaxed cost is guaranteed to be a lower bound on the real cost.

In [ ]:
# 5x5 grid; 1 = wall. Start at (0,0), goal at (4,4).
grid = np.zeros((5, 5), int)
wall_cells = [(1,0),(1,1),(1,2),(1,3),(3,1),(3,2),(3,3),(3,4)]
for r, c in wall_cells:
    grid[r, c] = 1

start, goal = (0, 0), (4, 4)

def manhattan(a, b):
    # Relaxed cost with walls removed: horizontal gap + vertical gap.
    row_gap = abs(a[0] - b[0])
    col_gap = abs(a[1] - b[1])
    return row_gap + col_gap

### Step 2 — Compute the true cost with BFS

To check the heuristic is **admissible** we need the actual shortest path through the maze, walls and all. A breadth-first search (BFS) explores outward layer by layer, so the first time it reaches the goal it has found the fewest-step path.

We expand each cell to its four neighbours, skipping walls, out-of-bounds cells, and cells already seen, recording the distance `d` as we go.

In [ ]:
from collections import deque

def true_cost(grid, s, g):
    # BFS for the shortest path that obeys the walls.
    queue = deque([(s, 0)])
    seen = {s}
    while queue:
        (r, c), d = queue.popleft()
        if (r, c) == g:
            return d
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            in_bounds = 0 <= nr < 5 and 0 <= nc < 5
            if in_bounds and grid[nr, nc] == 0 and (nr, nc) not in seen:
                seen.add((nr, nc))
                queue.append(((nr, nc), d + 1))
    return float("inf")

### Step 3 — Confirm the heuristic is admissible

Finally we compare the two numbers. The relaxed heuristic `h` should be **less than or equal to** the true maze cost — that is the admissibility condition A* relies on to stay optimal.

The walls force a longer detour, so the true cost is larger than the straight-line Manhattan guess, and `h <= cost` holds.

In [ ]:
h = manhattan(start, goal)
cost = true_cost(grid, start, goal)

print("relaxed heuristic h =", h)
print("true maze cost      =", cost)
print("admissible (h <= cost)?", h <= cost)

## Visualize the data & results

_Hospital delivery robot: how good is the relaxed (walls-removed) A* heuristic for the route to the Pharmacy?_

### Step 1 — Lay out the hospital floor and its heuristic field

A delivery robot must reach the **Pharmacy at (5,5)** through a serpentine corridor. We model the wing as a 6×6 grid with a set of wall cells.

The relaxed heuristic `H[r, c]` is the walls-ignored Manhattan distance from every cell to the pharmacy — a whole *field* of admissible lower bounds, one per cell.

In [ ]:
# Hospital 1st-floor wing as a 6x6 grid. Goal = Pharmacy at (5,5).
# Walls form a serpentine corridor between wards, labs and the pharmacy.
GRID, goal = 6, (5, 5)
walls = {(0,1),(1,1),(2,1),(3,1),(4,1),
         (1,3),(2,3),(3,3),(4,3),(5,3),
         (0,5),(1,5),(2,5),(3,5)}

# Relaxed heuristic: Manhattan steps to the Pharmacy with walls removed.
H = np.zeros((GRID, GRID), int)
for r in range(GRID):
    for c in range(GRID):
        H[r, c] = abs(r - goal[0]) + abs(c - goal[1])

### Step 2 — Run real A* on the walled grid

To see how much the walls cost the robot, we run a real A* search that *does* respect them, using the heuristic field `H` to prioritise its frontier (a priority queue ordered by `f = g + h`).

The straight-line guess from the start is `h = 10`, but the true walled route is `20` steps — the relaxation underestimates by half, yet never overestimates, so A* stays correct.

In [ ]:
import heapq

# Real A* on the walled grid confirms h(start)=10 vs true cost=20 (admissible).
def astar(start):
    pq = [(H[start], 0, start)]      # entries are (f = g + h, g, state)
    seen = {}
    while pq:
        f, g, s = heapq.heappop(pq)
        if s in seen and g >= seen[s]:
            continue                 # already reached this cell more cheaply
        seen[s] = g
        if s == goal:
            return g
        r, c = s
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            in_bounds = nr in range(GRID) and nc in range(GRID)
            if in_bounds and (nr, nc) not in walls:
                new_g = g + 1
                new_f = new_g + H[nr, nc]
                heapq.heappush(pq, (new_f, new_g, (nr, nc)))

print("h(start)=", H[0, 0], " true walled cost=", astar((0, 0)))

### Step 3 — Visualize the heuristic field

Finally we render `H` as a heatmap, annotating each cell with its distance-to-pharmacy estimate. The values form smooth contours sloping down toward the goal at (5,5), which is exactly what guides A* — even though the robot's real path must weave around the walls those numbers ignore.

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(H, cmap="viridis")

# Annotate each cell with its heuristic value.
for r in range(GRID):
    for c in range(GRID):
        ax.text(c, r, H[r, c], ha="center", va="center", color="w")

ax.set_title("Relaxed heuristic h = steps to Pharmacy (5,5), walls ignored")
fig.colorbar(im, ax=ax)
plt.show()